In [0]:
from pyspark.sql.functions import col, to_date, month, concat_ws, when, round, upper, expr, udf
from pyspark.sql.types import StringType, IntegerType, ArrayType
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from functools import reduce

In [0]:
# Databricks initialization notebook source 
try:
    verbose_mode = dbutils.widgets.get("verbose_mode");
except:
    verbose_mode = 'debug'
try:
    current_division = dbutils.widgets.get("division");
except:
    current_division = 'mal'
try:
    current_environment = dbutils.widgets.get("environment");
except:
    current_environment = 'dev' #dev
try:
    execution_mode = dbutils.widgets.get("execution_mode");
except:
    execution_mode = 'update'
try:
    current_project = dbutils.widgets.get("project");
except:
    current_project = 'maite_bi'
try:
    current_production_line = dbutils.widgets.get("production_line");
except:
    current_production_line = 'gains'

current_catalog = current_division + '_' + current_project + '_' + current_environment;

current_schema = current_production_line;

current_location = 'abfss://' + current_project + '@adlsdpcom'+ current_environment +f'data.dfs.core.windows.net/' + current_production_line + '/'

if verbose_mode == 'debug':
    display("Debug Mode")
    display(f"current_division : {current_division}")
    display(f"current_environment : {current_environment}")
    display(f"current_project : {current_project}")
    display(f"current_production_line : {current_production_line}")
    display(f"current_catalog : {current_catalog}")
    display(f"current_schema : {current_schema}")
    display(f"current_location : {current_location}")

In [0]:
if current_environment =='preprd':
    source = f"""ext_mal_psql_maite_vision_board_test.public"""    
else:
    source = f"""ext_mal_psql_maite_vision_board_{current_environment}.public"""

In [0]:
source_gold = f"""mal_maite_{current_environment}"""

In [0]:
source_temp = f"""mal_maite_bi_{current_environment}.gains"""

In [0]:
master_table_source = f"""
SELECT * from {source_gold}.rouen1.master_table UNION ALL
SELECT * from {source_gold}.nogent1.master_table UNION ALL
SELECT * from {source_gold}.nogent2.master_table UNION ALL
SELECT * from {source_gold}.prouvy1.master_table UNION ALL
SELECT * from {source_gold}.polisy1.master_table UNION ALL
SELECT * from {source_gold}.strasbourg2.master_table UNION ALL
SELECT * from {source_gold}.hodonice1.master_table UNION ALL
SELECT * from {source_gold}.burton1.master_table UNION ALL
SELECT * from {source_gold}.buzau1.master_table UNION ALL
SELECT * from {source_gold}.bolelemi1.master_table
"""

master_table_source = spark.sql(master_table_source)
master_table_source.createOrReplaceTempView("master_table_source")

In [0]:
master_table = f"""
SELECT mtp.batch_number AS batch,
      mtp.id_plant AS id_site_production,
      mtp.production_line AS site_production,
      dfp.done AS date_fin_production,
      mtp.production_type AS type_de_production,
      mtp.specifications_name AS cahier_des_charges,
      mtp.goods_specy,
      mtp.goods_variety,
      mtp.malt_yield_r2 AS reel_r2_humide,
      mtp.goods_weight AS reel_orge_humide,
      mtp.goods_moisture AS reel_humidite_orge,
      mtp.malt_weight AS reel_malt_humide,
      mtp.malt_moisture AS reel_humidite_malt
FROM master_table_source mtp
JOIN {source_temp}.date_fin_production_temp dfp ON dfp.batch_number = mtp.batch_number
WHERE mtp.batch_production_year >= 2024
"""

df_master_table = spark.sql(master_table)

In [0]:
# Appliquer la transformation pour convertir en majuscules la colonne 'batch_number'
df_master_table = df_master_table.withColumn("batch", upper(df_master_table["batch"]))

In [0]:
# Calcul du R2_sec
df_master_table = df_master_table.withColumn("reel_r2_sec",
(col("reel_malt_humide") - (col("reel_malt_humide") * (col("reel_humidite_malt") / 100))) /
(col("reel_orge_humide") - (col("reel_orge_humide") * (col("reel_humidite_orge") / 100)))* 100)

In [0]:
# Créer une vue temporaire pour le DataFrame
df_master_table.createOrReplaceTempView("master_table")

In [0]:
# Ajout des baseline par CDC, site de production et par mois
mt_baseline = f"""
SELECT
  mt.*,
  b.nb_individus,
  b.Baseline_malt_yield_r2,
  b.Baseline_malt_dry_yield,
  b.Baseline_goods_weight,
  b.Baseline_goods_moisture,
  b.Baseline_malt_moisture,
  b.Baseline_FAN,
  b.Baseline_friabilite,
  b.Baseline_coloration_EBC,
  b.Baseline_betaG,
  b.Baseline_quality
FROM master_table mt
JOIN {source_temp}.baseline b ON b.Specification = mt.cahier_des_charges AND b.id_site = mt.id_site_production AND b.Month = MONTH(mt.date_fin_production)
"""

df_mt_baseline = spark.sql(mt_baseline)

In [0]:
# Renommer les colonnes baseline
df_mt_baseline = df_mt_baseline.withColumnRenamed("nb_individus", "nb_individus_baseline")
df_mt_baseline = df_mt_baseline.withColumnRenamed("Baseline_malt_yield_r2", "baseline_r2_humide")
df_mt_baseline = df_mt_baseline.withColumnRenamed("Baseline_malt_dry_yield", "baseline_r2_sec")
df_mt_baseline = df_mt_baseline.withColumnRenamed("Baseline_goods_weight", "baseline_orge_humide")
df_mt_baseline = df_mt_baseline.withColumnRenamed("Baseline_goods_moisture", "baseline_humidite_orge")
df_mt_baseline = df_mt_baseline.withColumnRenamed("Baseline_malt_moisture", "baseline_humidite_malt")
df_mt_baseline = df_mt_baseline.withColumnRenamed("Baseline_quality", "baseline_indice_qualite")

In [0]:
# Création d'une nouvelle colonne 'type_de_baseline' basée sur la condition de 'nb_individus_baseline'
df_mt_baseline = df_mt_baseline.withColumn(
    'type_de_baseline',
    F.when(
        F.col('nb_individus_baseline') >= 5,
        F.lit("Nombre d’individus suffisant")
    ).otherwise(F.lit("Baseline comblée"))
)

In [0]:
columns = ['baseline_humidite_orge', 'baseline_r2_humide', 'baseline_r2_sec', 'baseline_humidite_malt',
           'reel_r2_humide', 'reel_r2_sec', 'reel_humidite_malt', 'reel_humidite_orge']

# Appliquer la division sur chaque colonne spécifiée
for column in columns:
    df_mt_baseline = df_mt_baseline.withColumn(column, col(column) / 100)

In [0]:
df_mt_baseline = df_mt_baseline.withColumn("baseline_orge_sec",col("baseline_orge_humide") - (col("baseline_orge_humide") * col("baseline_humidite_orge")))
df_mt_baseline = df_mt_baseline.withColumn("baseline_malt_sec",col("baseline_orge_sec") * col("baseline_r2_sec"))
df_mt_baseline = df_mt_baseline.withColumn("baseline_eau_dans_orge",col("baseline_orge_humide") * col("baseline_humidite_orge"))
df_mt_baseline = df_mt_baseline.withColumn("baseline_eau_dans_malt",col("baseline_malt_sec") / ( 1 - col("baseline_humidite_malt")) - col("baseline_malt_sec"))
df_mt_baseline = df_mt_baseline.withColumn("baseline_malt_humide",col("baseline_eau_dans_malt") + col("baseline_malt_sec"))

df_mt_baseline = df_mt_baseline.withColumn("reel_eau_dans_orge",col("reel_orge_humide") * col("reel_humidite_orge"))
df_mt_baseline = df_mt_baseline.withColumn("reel_orge_sec",col("reel_orge_humide") - (col("reel_orge_humide") * col("reel_humidite_orge")))
df_mt_baseline = df_mt_baseline.withColumn("reel_malt_sec",col("reel_orge_sec") * col("reel_r2_sec"))
df_mt_baseline = df_mt_baseline.withColumn("reel_ecart_r2_sec_comparaison_baseline",col("reel_r2_sec") - col("baseline_r2_sec"))
df_mt_baseline = df_mt_baseline.withColumn("reel_ecart_orge_sec_comparaison_baseline",col("reel_orge_sec") - col("baseline_orge_sec"))

df_mt_baseline = df_mt_baseline.withColumn("malt_sec_supplementaire_via_perf_r2_sec",col("reel_ecart_r2_sec_comparaison_baseline") * col("baseline_orge_sec"))
df_mt_baseline = df_mt_baseline.withColumn("malt_sec_supplementaire_via_taille_batch_equivalent_baseline",col("reel_ecart_orge_sec_comparaison_baseline") * col("baseline_r2_sec"))
df_mt_baseline = df_mt_baseline.withColumn("malt_sec_supplementaire_via_taille_batch_et_perf_r2_sec",
                             col("reel_ecart_r2_sec_comparaison_baseline") * col("reel_ecart_orge_sec_comparaison_baseline"))

df_mt_baseline = df_mt_baseline.withColumn("reel_eau_dans_malt",col("reel_malt_sec") / (1 - col("reel_humidite_malt")) - col("reel_malt_sec"))
df_mt_baseline = df_mt_baseline.withColumn("reel_malt_humide",col("reel_eau_dans_malt") + col("reel_malt_sec"))
df_mt_baseline = df_mt_baseline.withColumn("eau_equivalent_baseline",col("baseline_malt_sec") / (1 - col("baseline_humidite_malt")) - col("baseline_malt_sec"))

df_mt_baseline = df_mt_baseline.withColumn('eau_supplementaire_via_perf_humidite_malt',
                             col('baseline_malt_sec') / (1 - col('reel_humidite_malt')) - col('baseline_malt_sec') - col('eau_equivalent_baseline'))

df_mt_baseline = df_mt_baseline.withColumn('eau_supplementaire_via_perf_r2_sec_equivalent_baseline',
                             col('malt_sec_supplementaire_via_perf_r2_sec') / (1 - col('baseline_humidite_malt')) - col('malt_sec_supplementaire_via_perf_r2_sec'))

df_mt_baseline = df_mt_baseline.withColumn('eau_supplementaire_via_perf_r2_sec_et_perf_humidite_malt',
                             (col('malt_sec_supplementaire_via_perf_r2_sec') / (1 - col('reel_humidite_malt')) - col('malt_sec_supplementaire_via_perf_r2_sec')) -
                             col('eau_supplementaire_via_perf_r2_sec_equivalent_baseline'))

df_mt_baseline = df_mt_baseline.withColumn('eau_supplementaire_via_taille_batch_equivalent_baseline',
                             col('malt_sec_supplementaire_via_taille_batch_equivalent_baseline') / (1 - col('baseline_humidite_malt')) -
                             col('malt_sec_supplementaire_via_taille_batch_equivalent_baseline'))
                            
df_mt_baseline = df_mt_baseline.withColumn('eau_supplementaire_via_taille_batch_et_perf_humidite_malt',
                             (col('malt_sec_supplementaire_via_taille_batch_equivalent_baseline') / (1 - col('reel_humidite_malt')) -
                              col('malt_sec_supplementaire_via_taille_batch_equivalent_baseline')) - col('eau_supplementaire_via_taille_batch_equivalent_baseline'))

df_mt_baseline = df_mt_baseline.withColumn("eau_supplementaire_via_taille_batch_et_perf_r2_sec_equivalent_baseline",
                             col("malt_sec_supplementaire_via_taille_batch_et_perf_r2_sec") / (1 - col("baseline_humidite_malt")) -
                             col("malt_sec_supplementaire_via_taille_batch_et_perf_r2_sec"))

df_mt_baseline = df_mt_baseline.withColumn("eau_supplementaire_via_taille_batch_et_perf_r2_sec_et_perf_humidite_malt",
                             (col("malt_sec_supplementaire_via_taille_batch_et_perf_r2_sec") / (1 - col("reel_humidite_malt")) -
                              col("malt_sec_supplementaire_via_taille_batch_et_perf_r2_sec")) - col("eau_supplementaire_via_taille_batch_et_perf_r2_sec_equivalent_baseline"))

df_mt_baseline = df_mt_baseline.withColumn('R2/H - Malt_sec',col('malt_sec_supplementaire_via_perf_r2_sec') + col('malt_sec_supplementaire_via_taille_batch_et_perf_r2_sec'))

df_mt_baseline = df_mt_baseline.withColumn('R2/H - Eau',col('eau_supplementaire_via_perf_humidite_malt') + col('eau_supplementaire_via_perf_r2_sec_equivalent_baseline') + 
                             col('eau_supplementaire_via_perf_r2_sec_et_perf_humidite_malt') + col('eau_supplementaire_via_taille_batch_et_perf_humidite_malt') +
                             col('eau_supplementaire_via_taille_batch_et_perf_r2_sec_equivalent_baseline') + 
                             col('eau_supplementaire_via_taille_batch_et_perf_r2_sec_et_perf_humidite_malt'))

df_mt_baseline = df_mt_baseline.withColumn('total_malt_sec_supplementaire',
                             col('malt_sec_supplementaire_via_perf_r2_sec') + col('malt_sec_supplementaire_via_taille_batch_equivalent_baseline') +
                             col('malt_sec_supplementaire_via_taille_batch_et_perf_r2_sec'))

df_mt_baseline = df_mt_baseline.withColumn('total_eau_supplementaire',
                             col('eau_supplementaire_via_perf_humidite_malt') + col('eau_supplementaire_via_perf_r2_sec_equivalent_baseline') +
                             col('eau_supplementaire_via_perf_r2_sec_et_perf_humidite_malt') + col('eau_supplementaire_via_taille_batch_equivalent_baseline') + 
                             col('eau_supplementaire_via_taille_batch_et_perf_humidite_malt') +
                             col('eau_supplementaire_via_taille_batch_et_perf_r2_sec_equivalent_baseline') + 
                             col('eau_supplementaire_via_taille_batch_et_perf_r2_sec_et_perf_humidite_malt'))

df_mt_baseline = df_mt_baseline.withColumn('total_malt_humide_supplementaire',col('total_malt_sec_supplementaire') + col('total_eau_supplementaire'))

df_mt_baseline = df_mt_baseline.withColumn('gain_avec_IA',col('R2/H - Malt_sec') + col('R2/H - Eau') + col('eau_supplementaire_via_taille_batch_equivalent_baseline'))

In [0]:
# Créer une vue temporaire pour le DataFrame
df_mt_baseline.createOrReplaceTempView("mt_baseline")

INDICE QUALITÉ

In [0]:
bqt = f"""
SELECT b.batch_number, pe.code, btq.upper_limit, btq.lower_limit
FROM {source}.batches_quality_targets btq
JOIN {source}.batches b ON b.id_batch = btq.batch
JOIN {source}.parameters_evaluations pe ON btq.evaluation_parameter = pe.id_parameter_evaluation
"""

df_bqt = spark.sql(bqt)

In [0]:
# Max upper_limit
df_bqt_max = df_bqt.groupBy("batch_number").pivot("code").agg(F.max("upper_limit"))

# Renommer les colonnes dans df_bqt_max
df_bqt_max = df_bqt_max.withColumnRenamed('malt_color_ebc', 'coloration_EBC_max') \
                       .withColumnRenamed('malt_free_amino_nitrogen', 'FAN_max') \
                       .withColumnRenamed('malt_friability', 'friabilite_max') \
                       .withColumnRenamed('malt_soluble_betaglucan', 'betaG_max') \
                       .withColumnRenamed('product_moisture', 'hum_malt_max')

# Max lower_limit
df_bqt_min = df_bqt.groupBy("batch_number").pivot("code").agg(F.max("lower_limit"))

# Renommer les colonnes dans df_bqt_min
df_bqt_min = df_bqt_min.withColumnRenamed('malt_color_ebc', 'coloration_EBC_min') \
                       .withColumnRenamed('malt_free_amino_nitrogen', 'FAN_min') \
                       .withColumnRenamed('malt_friability', 'friabilite_min') \
                       .withColumnRenamed('malt_soluble_betaglucan', 'betaG_min') \
                       .withColumnRenamed('product_moisture', 'hum_malt_min')

# Jointure sur 'batch_number'
df_bqt = df_bqt_max.join(df_bqt_min.select(
    'batch_number', 
    'coloration_EBC_min', 
    'FAN_min', 
    'friabilite_min', 
    'betaG_min', 
    'hum_malt_min'
), on='batch_number', how='left')

# Sélection des colonnes finales
df_bqt = df_bqt.select(
    'batch_number',
    'FAN_max',
    'FAN_min',
    'hum_malt_max',
    'hum_malt_min',
    'betaG_max',
    'betaG_min',
    'friabilite_max',
    'friabilite_min',
    'coloration_EBC_max',
    'coloration_EBC_min'
)

In [0]:
# Créer une vue temporaire pour le DataFrame
df_bqt.createOrReplaceTempView("batches_quality_targets_min_max")

In [0]:
indice_qualite = f"""
SELECT
  bqt.*,
  mtp.malt_free_amino_nitrogen AS FAN,
  mtp.malt_moisture AS hum_malt,
  mtp.malt_color_ebc AS coloration_EBC,
  mtp.malt_friability AS friabilite,
  mtp.malt_soluble_beta_glucan AS betaG
FROM master_table_source mtp
JOIN batches_quality_targets_min_max bqt ON bqt.batch_number = mtp.batch_number
"""

df_indice_qualite = spark.sql(indice_qualite)

In [0]:
# Calcul de FAN_result
df_indice_qualite = df_indice_qualite.withColumn(
    'FAN_result', 
    F.when((F.col('FAN_min') <= F.col('FAN')) & (F.col('FAN') <= F.col('FAN_max')), 0.2).otherwise(0)
)

# Calcul de hum_malt_result
df_indice_qualite = df_indice_qualite.withColumn(
    'hum_malt_result', 
    F.when((F.col('hum_malt_min') <= F.col('hum_malt')) & (F.col('hum_malt') <= F.col('hum_malt_max')), 0.2).otherwise(0)
)

# Calcul de betaG_result
df_indice_qualite = df_indice_qualite.withColumn(
    'betaG_result', 
    F.when((F.col('betaG_min') <= F.col('betaG')) & (F.col('betaG') <= F.col('betaG_max')), 0.2).otherwise(0)
)

# Calcul de friabilite_result
df_indice_qualite = df_indice_qualite.withColumn(
    'friabilite_result', 
    F.when((F.col('friabilite_min') <= F.col('friabilite')) & (F.col('friabilite') <= F.col('friabilite_max')), 0.2).otherwise(0)
)

# Calcul de coloration_EBC_result
df_indice_qualite = df_indice_qualite.withColumn(
    'coloration_EBC_result', 
    F.when((F.col('coloration_EBC_min') <= F.col('coloration_EBC')) & (F.col('coloration_EBC') <= F.col('coloration_EBC_max')), 0.2).otherwise(0)
)

# Calcul de l'indice_qualite
df_indice_qualite = df_indice_qualite.withColumn(
    'indice_qualite', 
    F.round(
        F.col('FAN_result') + F.col('hum_malt_result') + F.col('betaG_result') + F.col('friabilite_result') + F.col('coloration_EBC_result'),1))

In [0]:
# Appliquer la transformation pour convertir en majuscules la colonne 'batch_number'
df_indice_qualite = df_indice_qualite.withColumn("batch_number", upper(df_indice_qualite["batch_number"]))

In [0]:
df_indice_qualite = df_indice_qualite.select(
    'batch_number',
    'FAN', 'FAN_max', 'FAN_min', 'FAN_result',
    'hum_malt', 'hum_malt_max', 'hum_malt_min', 'hum_malt_result',
    'betaG', 'betaG_max', 'betaG_min', 'betaG_result',
    'friabilite', 'friabilite_max', 'friabilite_min', 'friabilite_result',
    'coloration_EBC', 'coloration_EBC_max', 'coloration_EBC_min', 'coloration_EBC_result',
    'indice_qualite'
)

In [0]:
# Créer une vue temporaire pour le DataFrame
df_indice_qualite.createOrReplaceTempView("indice_qualite")

RECOMMANDATIONS

In [0]:
fact = f"""
SELECT mtb.*,
        iq.FAN, iq.FAN_max, iq.FAN_min, iq.FAN_result,
        iq.hum_malt, iq.hum_malt_max, iq.hum_malt_min, iq.hum_malt_result,
        iq.betaG, iq.betaG_max, iq.betaG_min, iq.betaG_result,
        iq.friabilite, iq.friabilite_max, iq.friabilite_min, iq.friabilite_result,
        iq.coloration_EBC, iq.coloration_EBC_max, iq.coloration_EBC_min, iq.coloration_EBC_result,
        iq.indice_qualite,
        a.reco_acceptee, t.reco_trempe, g.reco_germination, to.reco_touraille
FROM mt_baseline mtb
JOIN indice_qualite iq ON iq.batch_number = mtb.batch
JOIN master_table_source mt ON mt.batch_number = mtb.batch

LEFT JOIN (SELECT b.batch_number, SUM(r.acceptance_status) AS reco_acceptee
        FROM {source}.recommendations r
        JOIN {source}.batches b ON b.id_batch = r.batch
        WHERE r.acceptance_status = 1
        GROUP BY b.batch_number) a ON a.batch_number = mt.batch_number

LEFT JOIN (SELECT b.batch_number, SUM(r.acceptance_status) AS reco_trempe
        FROM {source}.recommendations r
        JOIN {source}.batches b ON b.id_batch = r.batch
        JOIN {source}.parameters_localizations_translations plt ON plt.id_parameter_localization = r.target_localization
        WHERE plt.language = 1 AND plt.label LIKE 'Trempe%' AND r.acceptance_status = 1
        GROUP BY b.batch_number) t ON t.batch_number = mt.batch_number

LEFT JOIN (SELECT b.batch_number, SUM(r.acceptance_status) AS reco_germination
        FROM {source}.recommendations r
        JOIN {source}.batches b ON b.id_batch = r.batch
        JOIN {source}.parameters_localizations_translations plt ON plt.id_parameter_localization = r.target_localization
        WHERE plt.language = 1 AND plt.label LIKE 'Germination%' AND r.acceptance_status = 1
        GROUP BY b.batch_number) g ON g.batch_number = mt.batch_number

LEFT JOIN (SELECT b.batch_number, SUM(r.acceptance_status) AS reco_touraille
        FROM {source}.recommendations r
        JOIN {source}.batches b ON b.id_batch = r.batch
        JOIN {source}.parameters_localizations_translations plt ON plt.id_parameter_localization = r.target_localization
        WHERE plt.language = 1 AND plt.label LIKE 'Touraille%' AND r.acceptance_status = 1
        GROUP BY b.batch_number) to ON to.batch_number = mt.batch_number

WHERE mt.batch_production_year >= 2024
ORDER BY mt.batch_number
"""

df_fact = spark.sql(fact)

In [0]:
# Renommer les colonnes dans df_bqt_min
df_fact = df_fact.withColumnRenamed('Baseline_FAN', 'baseline_FAN') \
                .withColumnRenamed('Baseline_betaG', 'baseline_betaG') \
                .withColumnRenamed('Baseline_coloration_EBC', 'baseline_coloration_EBC') \
                .withColumnRenamed('Baseline_friabilite', 'baseline_friabilite')

In [0]:
# Sélection des colonnes spécifiques en PySpark
df_fact = df_fact.select(
'batch',
'date_fin_production',
'id_site_production',
'site_production',
'cahier_des_charges',
'type_de_production',
'goods_specy',
'goods_variety',
'nb_individus_baseline',
'type_de_baseline',
'baseline_r2_sec',
'baseline_r2_humide',
'baseline_orge_humide',
'baseline_humidite_orge',
'baseline_orge_sec',
'baseline_eau_dans_orge',
'baseline_malt_humide',
'baseline_humidite_malt',
'baseline_malt_sec',
'baseline_eau_dans_malt',
'reel_r2_sec',
'reel_r2_humide',
'reel_orge_humide',
'reel_humidite_orge',
'reel_orge_sec',
'reel_eau_dans_orge',
'reel_malt_humide',
'reel_humidite_malt',
'reel_malt_sec',
'reel_eau_dans_malt',
'reel_ecart_r2_sec_comparaison_baseline',
'reel_ecart_orge_sec_comparaison_baseline',
'malt_sec_supplementaire_via_perf_r2_sec',
'malt_sec_supplementaire_via_taille_batch_equivalent_baseline',
'malt_sec_supplementaire_via_taille_batch_et_perf_r2_sec',
'eau_equivalent_baseline',
'eau_supplementaire_via_perf_humidite_malt',
'eau_supplementaire_via_perf_r2_sec_equivalent_baseline',
'eau_supplementaire_via_perf_r2_sec_et_perf_humidite_malt',
'eau_supplementaire_via_taille_batch_equivalent_baseline',
'eau_supplementaire_via_taille_batch_et_perf_humidite_malt',
'eau_supplementaire_via_taille_batch_et_perf_r2_sec_equivalent_baseline',
'eau_supplementaire_via_taille_batch_et_perf_r2_sec_et_perf_humidite_malt',
'total_malt_sec_supplementaire',
'total_eau_supplementaire',
'total_malt_humide_supplementaire',
'R2/H - Malt_sec',
'R2/H - Eau',
'gain_avec_IA',
'reco_acceptee',
'reco_trempe',
'reco_germination',
'reco_touraille',
'FAN',
'FAN_max',
'FAN_min',
'FAN_result',
'baseline_FAN',
'betaG',
'betaG_max',
'betaG_min',
'betaG_result',
'baseline_betaG',
'friabilite',
'friabilite_max',
'friabilite_min',
'friabilite_result',
'baseline_friabilite',
'coloration_EBC',
'coloration_EBC_max',
'coloration_EBC_min',
'coloration_EBC_result',
'baseline_coloration_EBC',
'hum_malt',
'hum_malt_max',
'hum_malt_min',
'hum_malt_result',
'indice_qualite',
'baseline_indice_qualite'
)

ERREUR FACT

In [0]:
df_erreur_fact = df_fact

# Identification des erreurs et manques de données dans les colonnes spécifiées
colonnes_erreur = ['id_site_production', 'site_production', 'date_fin_production', 'type_de_production', 
                   'cahier_des_charges', 'baseline_orge_humide', 'baseline_r2_humide', 'baseline_humidite_orge', 
                   'baseline_r2_sec', 'baseline_humidite_malt', 'reel_orge_humide', 'reel_humidite_orge', 
                   'reel_humidite_malt', 'reel_malt_humide']

# Filtrer les lignes avec des données manquantes dans les colonnes spécifiées
condition = reduce(lambda x, y: x | y, [F.col(col).isNull() | (F.col(col) == '') for col in colonnes_erreur])
df_erreur_fact = df_erreur_fact.filter(condition)

# Création de la colonne 'causes' en listant les colonnes vides pour chaque ligne
df_erreur_fact = df_erreur_fact.withColumn(
    'causes',
    F.array([F.when(F.col(col).isNull() | (F.col(col) == ''), F.lit(f"{col} est vide")).otherwise(F.lit(None)) for col in colonnes_erreur])
)

# Supprimer les éléments vides de la liste 'causes'
df_erreur_fact = df_erreur_fact.withColumn('causes', F.expr("filter(causes, x -> x is not null)"))

# Explosion de la colonne 'causes'
df_erreur_fact = df_erreur_fact.withColumn('cause', F.explode('causes'))

# Filtrer les lignes où 'causes' commence par "Baseline" et 'date_fin_production' est vide
df_erreur_fact = df_erreur_fact.filter(~((F.col('cause').startswith('baseline')) & F.col('date_fin_production').isNull()))

# Sélectionner les colonnes finales
colonnes_finales = ['batch', 'id_site_production', 'site_production', 'date_fin_production', 'type_de_production', 
                    'cahier_des_charges', 'baseline_orge_humide', 'baseline_r2_humide', 'baseline_humidite_orge', 
                    'baseline_r2_sec', 'baseline_humidite_malt', 'reel_orge_humide', 'reel_humidite_orge', 
                    'reel_humidite_malt', 'reel_malt_humide', 'cause']
df_erreur_fact = df_erreur_fact.select(colonnes_finales)

# Supprimer les doublons dans le DataFrame
df_erreur_fact = df_erreur_fact.dropDuplicates()

In [0]:
# Supprimer les lignes où une ou plusieurs colonnes dans 'colonnes' ont des valeurs manquantes
df_fact = df_fact.dropna(subset=colonnes_erreur)

In [0]:
# Simulation Baseline Seche Gain IA V6 - 3 ans
display(df_fact)

In [0]:
# Trie croissant sur les colonnes suivantes
df_erreur_fact = df_erreur_fact.orderBy("id_site_production", "batch")

# fact_erreur
display(df_erreur_fact)

In [0]:
# Créer une vue temporaire pour le DataFrame
df_fact.createOrReplaceTempView("fact")

ENERGIES

In [0]:
nrj = f"""
SELECT
    f.batch, f.date_fin_production, f.cahier_des_charges, f.type_de_production, f.id_site_production, f.site_production, f.goods_specy, f.goods_variety,
    cet.global_electric_energy_lhv_consumption, cet.global_thermal_energy_lhv_consumption, b.Baseline_electricity, b.Baseline_thermal,
    f.reel_r2_sec, f.reel_orge_humide, f.reel_humidite_orge, f.reel_humidite_malt
FROM {source_temp}.conso_energie_temp cet
JOIN fact f ON f.batch = cet.batch_number
JOIN {source_temp}.baseline b ON b.Specification = f.cahier_des_charges AND b.id_site = f.id_site_production AND b.Month = MONTH(f.date_fin_production)
"""

df_nrj = spark.sql(nrj)

In [0]:
# Dictionnaire des colonnes à renommer
colonnes = {
    'global_electric_energy_lhv_consumption': 'reel_elec_kWh_t',
    'global_thermal_energy_lhv_consumption': 'reel_thermique_kWh_t'
}

# Conversion des colonnes en valeurs numériques et traitement des NaN et des valeurs inférieures à 10
for colonne_originale, nouveau_nom in colonnes.items():
    df_nrj = df_nrj.withColumn(
        colonne_originale, 
        F.when(F.col(colonne_originale).cast('float') < 10, 0)
        .otherwise(F.col(colonne_originale).cast('float'))
    )
    # Remplacement des NaN par 0
    df_nrj = df_nrj.fillna({colonne_originale: 0})
    # Renommage des colonnes
    df_nrj = df_nrj.withColumnRenamed(colonne_originale, nouveau_nom)

columns = ['reel_r2_sec', 'reel_humidite_orge', 'reel_humidite_malt']

# Calcul de 'goods_dry_weight'
df_nrj = df_nrj.withColumn(
    'reel_orge_sec', 
    F.col('reel_orge_humide') - (F.col('reel_orge_humide') * F.col('reel_humidite_orge'))
)

# Calcul de 'malt_dry_weight'
df_nrj = df_nrj.withColumn(
    'reel_malt_sec', 
    F.col('reel_orge_humide') * F.col('reel_r2_sec')
)

# Calcul de 'reel_eau_dans_malt'
df_nrj = df_nrj.withColumn(
    'reel_eau_dans_malt', 
    (F.col('reel_malt_sec') / (1 - F.col('reel_humidite_malt'))) - F.col('reel_malt_sec')
)

# Calcul de 'malt_weight'
df_nrj = df_nrj.withColumn(
    'reel_malt_humide', 
    F.col('reel_eau_dans_malt') + F.col('reel_malt_sec')
)

In [0]:
colonnes = {
    'Baseline_electricity': 'baseline_elec_kWh_t',
    'Baseline_thermal': 'baseline_thermique_kWh_t'
}

# Renommer les colonnes
for colonne_originale, nouveau_nom in colonnes.items():
    df_nrj = df_nrj.withColumnRenamed(colonne_originale, nouveau_nom)

# Calcul des écarts
df_nrj = df_nrj.withColumn('ecart_thermique_kWh_t', col('reel_thermique_kWh_t') - col('baseline_thermique_kWh_t'))
df_nrj = df_nrj.withColumn('ecart_elec_kWh_t', col('reel_elec_kWh_t') - col('baseline_elec_kWh_t'))

# Appliquer la condition where
df_nrj = df_nrj.withColumn('ecart_elec_kWh_t', 
                           when(col('reel_elec_kWh_t') <= 0, 0).otherwise(col('ecart_elec_kWh_t')))


In [0]:
# Sélection des colonnes spécifiques
df_nrj = df_nrj.select(
    'batch',
    'date_fin_production',
    'cahier_des_charges',
    'type_de_production',
    'id_site_production',
    'site_production',
    'goods_specy',
    'goods_variety',
    'reel_malt_humide',
    'reel_thermique_kWh_t',
    'baseline_thermique_kWh_t',
    'ecart_thermique_kWh_t',
    'reel_elec_kWh_t',
    'baseline_elec_kWh_t',
    'ecart_elec_kWh_t'
)

In [0]:
df_erreur_nrj = df_nrj

# Identification des erreurs et manques de données dans les colonnes spécifiées
colonnes_erreur_nrj = ['reel_thermique_kWh_t', 'baseline_thermique_kWh_t', 'reel_elec_kWh_t', 'baseline_elec_kWh_t']

# Construire la condition pour vérifier les valeurs manquantes
condition = None
for col in colonnes_erreur_nrj:
    if condition is None:
        condition = F.col(col).isNull() | (F.col(col) == '')
    else:
        condition = condition | (F.col(col).isNull() | (F.col(col) == ''))

# Filtrer les lignes avec des données manquantes dans les colonnes spécifiées
df_erreur_nrj = df_erreur_nrj.filter(condition)

# Création de la colonne 'causes' en listant les colonnes vides pour chaque ligne
df_erreur_nrj = df_erreur_nrj.withColumn(
    'causes',
    F.array([F.when(F.col(col).isNull() | (F.col(col) == ''), F.lit(f"{col} est vide")).otherwise(F.lit(None)) for col in colonnes_erreur_nrj])
)

# Supprimer les éléments nuls de la liste 'causes'
df_erreur_nrj = df_erreur_nrj.withColumn('causes', F.expr("filter(causes, x -> x is not null)"))

# Explosion de la colonne 'causes'
df_erreur_nrj = df_erreur_nrj.withColumn('cause', F.explode('causes'))

# Filtrer les lignes où 'causes' commence par "Baseline" et 'date_fin_production' est vide
df_erreur_nrj = df_erreur_nrj.filter(~((F.col('cause').startswith('baseline')) & F.col('date_fin_production').isNull()))

# Sélectionner les colonnes finales
colonnes_finales = ['batch', 'id_site_production', 'site_production', 'date_fin_production', 'type_de_production', 
                    'cahier_des_charges', 'reel_thermique_kWh_t', 'baseline_thermique_kWh_t', 'reel_elec_kWh_t', 
                    'baseline_elec_kWh_t', 'cause']
df_erreur_nrj = df_erreur_nrj.select(colonnes_finales)


In [0]:
# Supprimer les lignes où une ou plusieurs colonnes dans 'colonnes' ont des valeurs manquantes
df_erreur_nrj = df_erreur_nrj.dropna(subset=colonnes_erreur_nrj)

In [0]:
# Gain_Energie_kWh_t
display(df_nrj)

In [0]:
from pyspark.sql.functions import col

# Création d'un dataframe df_MWh qui est une "copie" de df_nrj
df_MWh = df_nrj

# Calcul des valeurs en MWh pour les colonnes thermique et électrique
df_MWh = df_MWh.withColumn('reel_thermique_MWh', (col('reel_thermique_kWh_t') * col('reel_malt_humide')) / 1000)
df_MWh = df_MWh.withColumn('reel_elec_MWh', (col('reel_elec_kWh_t') * col('reel_malt_humide')) / 1000)
df_MWh = df_MWh.withColumn('baseline_thermique_MWh', (col('baseline_thermique_kWh_t') * col('reel_malt_humide')) / 1000)
df_MWh = df_MWh.withColumn('baseline_elec_MWh', (col('baseline_elec_kWh_t') * col('reel_malt_humide')) / 1000)

# Calcul des écarts en MWh
df_MWh = df_MWh.withColumn('ecart_thermique_MWh', col('reel_thermique_MWh') - col('baseline_thermique_MWh'))
df_MWh = df_MWh.withColumn('ecart_elec_MWh', col('reel_elec_MWh') - col('baseline_elec_MWh'))

# Sélection des colonnes spécifiques
df_MWh = df_MWh.select(
    'batch',
    'date_fin_production',
    'cahier_des_charges',
    'type_de_production',
    'id_site_production',
    'site_production',
    'goods_specy',
    'goods_variety',
    'reel_malt_humide',
    'reel_thermique_MWh',
    'baseline_thermique_MWh',
    'ecart_thermique_MWh',
    'reel_elec_MWh',
    'baseline_elec_MWh',
    'ecart_elec_MWh'
)

In [0]:
# Gain_Energie_MWh
display(df_MWh)

In [0]:
# erreur_nrj
display(df_erreur_nrj)